# Byte-Level BPE vs. Character-Level (Normal) BPE

### A hands-on, from-scratch comparison

**Audience:** students who already know *what* tokenization is and have seen
Byte Pair Encoding (BPE) described at a high level ("merge the most frequent
adjacent pair, repeat").

**Goal of this notebook:** answer one very specific question —

> *When people say GPT-2/GPT-3/GPT-4 use "byte-level BPE", what exactly is
> different from the "normal" BPE algorithm described in the original 2016
> paper (Sennrich et al., *Neural Machine Translation of Rare Words with
> Subword Units*)?*

The surprising answer: **the merging algorithm itself does not change at
all.** Only one thing changes — *what counts as an atomic symbol before any
merging starts.* Everything else (count pairs → merge the most frequent pair
→ repeat) is identical.

We will:
1. Implement the shared BPE engine **once**.
2. Run it in **character mode** (the "normal"/classic version).
3. Run the *exact same engine* in **byte mode**.
4. Break the character version with an unseen symbol, and show the byte
   version can never be broken that way.
5. Confirm real production tokenizers (`tiktoken`, used by GPT-2/3.5/4) are
   byte-level, and see it with our own eyes on emojis, Hindi text, and math
   symbols.


## 1. The BPE engine — shared by both versions

Recall the algorithm, regardless of what a "symbol" is:

1. Start with a sequence of atomic symbols for every word in the training
   corpus.
2. Count how often every **adjacent pair** of symbols occurs, across the
   whole corpus.
3. Take the **single most frequent pair** and merge it into one new symbol.
   Add that new symbol to the vocabulary.
4. Repeat steps 2–3 until you hit your target vocabulary size (or no pair
   occurs more than once).

Nothing here mentions "character" or "byte". The engine below is written to
be agnostic to that choice — it just operates on a list of **symbols**,
whatever those symbols happen to be. We'll feed it two different kinds of
atoms later.


In [ ]:
from collections import Counter

def get_pair_counts(corpus_symbols):
    '''
    corpus_symbols: list of words, where each word is itself a list of symbols.
    e.g. [['l','o','w'], ['l','o','w','e','r']]

    Returns a Counter mapping (symbol_a, symbol_b) -> number of times that
    adjacent pair occurs, summed across every word in the corpus.
    '''
    pairs = Counter()
    for word in corpus_symbols:
        for a, b in zip(word[:-1], word[1:]):
            pairs[(a, b)] += 1
    return pairs


def merge_pair_in_corpus(corpus_symbols, pair_to_merge):
    '''
    Walks every word and replaces every occurrence of pair_to_merge=(a, b)
    with the single merged symbol (a + b), stitched together.
    '''
    a, b = pair_to_merge
    merged_symbol = a + b
    new_corpus = []
    for word in corpus_symbols:
        new_word = []
        i = 0
        while i < len(word):
            if i < len(word) - 1 and word[i] == a and word[i + 1] == b:
                new_word.append(merged_symbol)
                i += 2
            else:
                new_word.append(word[i])
                i += 1
        new_corpus.append(new_word)
    return new_corpus


def train_bpe(corpus_symbols, num_merges, verbose=True):
    '''
    The ENGINE. Identical for character-level and byte-level BPE.
    Returns:
        corpus_symbols  -> the corpus, fully merged
        merges          -> ordered list of merge rules learned, e.g. [(('l','o'), 'lo'), ...]
    '''
    merges = []
    for step in range(1, num_merges + 1):
        pair_counts = get_pair_counts(corpus_symbols)
        if not pair_counts:
            if verbose:
                print(f"Step {step}: no more pairs to merge. Stopping early.")
            break

        best_pair, count = pair_counts.most_common(1)[0]
        corpus_symbols = merge_pair_in_corpus(corpus_symbols, best_pair)
        merges.append((best_pair, best_pair[0] + best_pair[1]))

        if verbose:
            print(f"Step {step:>2}: merged {best_pair} -> {best_pair[0]+best_pair[1]!r}  "
                  f"(seen {count} times)")

    return corpus_symbols, merges

print("BPE engine defined. This exact code will be reused for BOTH experiments below.")


BPE engine defined. This exact code will be reused for BOTH experiments below.


## 2. Experiment A — "Normal" / classic character-level BPE

Here, the **atomic symbol = one Unicode character**. Before any merging, the
vocabulary is: *the set of unique characters that appear in the training
corpus* — nothing more, nothing less.

Toy corpus (word, frequency) — a classic tokenization teaching example:


In [ ]:
toy_corpus = {
    "low":     5,
    "lower":   2,
    "newest":  6,
    "widest":  3,
}

# Expand frequency into repeated words, and split each word into CHARACTERS.
# This is the "normal" BPE starting point: atoms = characters.
char_corpus = []
for word, freq in toy_corpus.items():
    char_corpus.extend([list(word)] * freq)

print("Training corpus (as character sequences):")
for word in sorted(set(tuple(w) for w in char_corpus)):
    print(" ", word)

initial_char_vocab = sorted(set(ch for word in char_corpus for ch in word))
print(f"\nInitial (base) vocabulary size = {len(initial_char_vocab)} characters")
print("Initial vocabulary:", initial_char_vocab)


Training corpus (as character sequences):
  ('l', 'o', 'w')
  ('l', 'o', 'w', 'e', 'r')
  ('n', 'e', 'w', 'e', 's', 't')
  ('w', 'i', 'd', 'e', 's', 't')

Initial (base) vocabulary size = 10 characters
Initial vocabulary: ['d', 'e', 'i', 'l', 'n', 'o', 'r', 's', 't', 'w']


In [ ]:
print("="*70)
print("TRAINING CHARACTER-LEVEL BPE")
print("="*70)

final_char_corpus, char_merges = train_bpe(char_corpus, num_merges=8)

print("\nFinal vocabulary after merges:")
final_char_vocab = set(initial_char_vocab)
for _, merged in char_merges:
    final_char_vocab.add(merged)
print(sorted(final_char_vocab, key=len))

print(f"\nTotal vocabulary size now: {len(final_char_vocab)} "
      f"({len(initial_char_vocab)} base characters + {len(char_merges)} learned merges)")


TRAINING CHARACTER-LEVEL BPE
Step  1: merged ('e', 's') -> 'es'  (seen 9 times)
Step  2: merged ('es', 't') -> 'est'  (seen 9 times)
Step  3: merged ('l', 'o') -> 'lo'  (seen 7 times)
Step  4: merged ('lo', 'w') -> 'low'  (seen 7 times)
Step  5: merged ('n', 'e') -> 'ne'  (seen 6 times)
Step  6: merged ('ne', 'w') -> 'new'  (seen 6 times)
Step  7: merged ('new', 'est') -> 'newest'  (seen 6 times)
Step  8: merged ('w', 'i') -> 'wi'  (seen 3 times)

Final vocabulary after merges:
['w', 'i', 'r', 't', 'o', 'n', 'l', 'e', 'd', 's', 'wi', 'ne', 'es', 'lo', 'est', 'low', 'new', 'newest']

Total vocabulary size now: 18 (10 base characters + 8 learned merges)


### 2.1 The weak point of character-level BPE

The base vocabulary was built **only from characters seen during training**.
What happens if, at inference time, someone hands the tokenizer a character
it has *never* seen before — say, an emoji, or a Chinese character, if the
training data was pure English?

There is no rule to fall back on. Classic implementations solve this with a
special `<UNK>` (unknown) token, which **throws away information** — the
model has no idea what that character actually was.

Let's simulate that failure honestly:


In [ ]:
def encode_char_bpe(word, merges, base_vocab):
    '''
    Encodes a single word using the character-level BPE we just trained.
    Any character NOT in base_vocab cannot even be represented as a starting
    symbol, so we mark it explicitly as unknown.
    '''
    symbols = list(word)

    # Check every character against what the tokenizer has ever seen.
    unknown_chars = [ch for ch in symbols if ch not in base_vocab]
    if unknown_chars:
        print(f"  !! Character(s) {unknown_chars} were NEVER seen during training.")
        print(f"  !! Classic BPE has no way to represent them -> falls back to <UNK>.")
        return None

    # Apply learned merges in the order they were learned.
    for (a, b), merged in merges:
        new_symbols = []
        i = 0
        while i < len(symbols):
            if i < len(symbols) - 1 and symbols[i] == a and symbols[i + 1] == b:
                new_symbols.append(merged)
                i += 2
            else:
                new_symbols.append(symbols[i])
                i += 1
        symbols = new_symbols
    return symbols


print("Encoding a normal, in-vocabulary word: 'lowest'")
result = encode_char_bpe("lowest", char_merges, initial_char_vocab)
print("  Tokens:", result)

print("\nEncoding a word containing an UNSEEN character: 'low🙂'")
result = encode_char_bpe("low🙂", char_merges, initial_char_vocab)
print("  Result:", result)


Encoding a normal, in-vocabulary word: 'lowest'
  Tokens: ['low', 'est']

Encoding a word containing an UNSEEN character: 'low🙂'
  !! Character(s) ['🙂'] were NEVER seen during training.
  !! Classic BPE has no way to represent them -> falls back to <UNK>.
  Result: None


**This is the core limitation.** Character-level BPE's base alphabet is
*whatever characters happened to appear in the training set*. Unicode has
over 150,000 possible characters (emoji, every world script, symbols, ...).
No training corpus contains all of them, so `<UNK>` is always a risk in
production — and every time it fires, information is silently destroyed.


## 3. Experiment B — Byte-level BPE

Now we make **exactly one change** to the recipe: instead of splitting words
into Unicode *characters*, we first encode each word as raw **UTF-8 bytes**,
and split into *those*.

Why does this matter? Because **every possible piece of text in every
language, emoji included, is made of some sequence of bytes from 0–255.**
That is what UTF-8 guarantees. So if our atomic alphabet is "the 256 possible
byte values", the alphabet is *complete by construction* — there is no such
thing as a byte the tokenizer has never heard of, because we don't even need
to have seen it: 0–255 is the entire universe of possibilities, decided
upfront, not learned from data.

Everything else — `get_pair_counts`, `merge_pair_in_corpus`, `train_bpe` — is
the **exact same code** we already wrote above.


In [ ]:
def word_to_bytes_symbols(word):
    '''
    THIS is the one-line algorithmic difference between byte-level and
    character-level BPE. Instead of list(word) [-> characters], we do:
    '''
    utf8_bytes = word.encode("utf-8")          # str -> raw bytes
    return [bytes([b]) for b in utf8_bytes]    # split into single-byte symbols


# Same toy corpus as before, but now split into BYTES instead of characters.
byte_corpus = []
for word, freq in toy_corpus.items():
    byte_corpus.extend([word_to_bytes_symbols(word)] * freq)

print("Training corpus (as byte sequences) -- compare this to the character version:")
for word in sorted(set(tuple(w) for w in byte_corpus)):
    print(" ", [b for b in word])

print(f"\nBase vocabulary: ALL 256 possible byte values ALWAYS, whether or not")
print(f"they showed up in this particular corpus. Base vocabulary size = 256.")
print(f"(Compare: the character version's base vocab size was only "
      f"{len(initial_char_vocab)}, and depended entirely on this corpus.)")


Training corpus (as byte sequences) -- compare this to the character version:
  [b'l', b'o', b'w']
  [b'l', b'o', b'w', b'e', b'r']
  [b'n', b'e', b'w', b'e', b's', b't']
  [b'w', b'i', b'd', b'e', b's', b't']

Base vocabulary: ALL 256 possible byte values ALWAYS, whether or not
they showed up in this particular corpus. Base vocabulary size = 256.
(Compare: the character version's base vocab size was only 10, and depended entirely on this corpus.)


In [ ]:
print("="*70)
print("TRAINING BYTE-LEVEL BPE")
print("="*70)

final_byte_corpus, byte_merges = train_bpe(byte_corpus, num_merges=8)

print("\nLearned merges (byte-level):")
for (a, b), merged in byte_merges:
    # decode just for pretty-printing -- the tokenizer itself never assumes
    # a merged chunk of bytes is valid, printable UTF-8.
    try:
        pretty = merged.decode("utf-8")
    except UnicodeDecodeError:
        pretty = merged
    print(f"  {a} + {b} -> {merged}  (looks like: {pretty!r})")


TRAINING BYTE-LEVEL BPE
Step  1: merged (b'e', b's') -> b'es'  (seen 9 times)
Step  2: merged (b'es', b't') -> b'est'  (seen 9 times)
Step  3: merged (b'l', b'o') -> b'lo'  (seen 7 times)
Step  4: merged (b'lo', b'w') -> b'low'  (seen 7 times)
Step  5: merged (b'n', b'e') -> b'ne'  (seen 6 times)
Step  6: merged (b'ne', b'w') -> b'new'  (seen 6 times)
Step  7: merged (b'new', b'est') -> b'newest'  (seen 6 times)
Step  8: merged (b'w', b'i') -> b'wi'  (seen 3 times)

Learned merges (byte-level):
  b'e' + b's' -> b'es'  (looks like: 'es')
  b'es' + b't' -> b'est'  (looks like: 'est')
  b'l' + b'o' -> b'lo'  (looks like: 'lo')
  b'lo' + b'w' -> b'low'  (looks like: 'low')
  b'n' + b'e' -> b'ne'  (looks like: 'ne')
  b'ne' + b'w' -> b'new'  (looks like: 'new')
  b'new' + b'est' -> b'newest'  (looks like: 'newest')
  b'w' + b'i' -> b'wi'  (looks like: 'wi')


### 3.1 Now try the SAME unseen input that broke character-level BPE


In [ ]:
def encode_byte_bpe(word, merges):
    '''
    Encodes a word with byte-level BPE. Notice: there is no 'unknown
    character' branch at all, because word_to_bytes_symbols(word) ALWAYS
    succeeds for ANY string in ANY language. That's the whole point.
    '''
    symbols = word_to_bytes_symbols(word)
    for (a, b), merged in merges:
        new_symbols = []
        i = 0
        while i < len(symbols):
            if i < len(symbols) - 1 and symbols[i] == a and symbols[i + 1] == b:
                new_symbols.append(merged)
                i += 2
            else:
                new_symbols.append(symbols[i])
                i += 1
        symbols = new_symbols
    return symbols


print("Encoding a normal, in-vocabulary word: 'lowest'")
print("  Tokens:", encode_byte_bpe("lowest", byte_merges))

print("\nEncoding the SAME unseen input that broke character-level BPE: 'low🙂'")
tokens = encode_byte_bpe("low🙂", byte_merges)
print("  Tokens:", tokens)
print("  No <UNK>, no crash. The emoji just wasn't part of any *learned merge*,")
print("  so it falls all the way back to its raw, individual UTF-8 bytes --")
print("  which were ALWAYS in the vocabulary, since day one.")

# Prove it round-trips perfectly, byte for byte.
rebuilt = b"".join(tokens).decode("utf-8")
print(f"\n  Decoded back: {rebuilt!r}  ->  matches original: {rebuilt == 'low🙂'}")


Encoding a normal, in-vocabulary word: 'lowest'
  Tokens: [b'low', b'est']

Encoding the SAME unseen input that broke character-level BPE: 'low🙂'
  Tokens: [b'low', b'\xf0', b'\x9f', b'\x99', b'\x82']
  No <UNK>, no crash. The emoji just wasn't part of any *learned merge*,
  so it falls all the way back to its raw, individual UTF-8 bytes --
  which were ALWAYS in the vocabulary, since day one.

  Decoded back: 'low🙂'  ->  matches original: True


**That's the entire algorithmic difference**, stated precisely:

| | Character-level ("normal") BPE | Byte-level BPE |
|---|---|---|
| Atomic symbol | 1 Unicode character | 1 raw byte (0–255) |
| Base vocabulary size | Variable — however many unique characters appear in training data | Always exactly 256 |
| Can it ever meet a symbol it has no representation for? | **Yes** — any character absent from training → `<UNK>` | **No** — every string is bytes, and all 256 byte values are always in the vocabulary |
| Merge algorithm (count pairs, merge most frequent, repeat) | Identical | Identical |
| Used in production by | Original BPE paper (Sennrich et al., 2016), older NMT systems | GPT-2, GPT-3, GPT-3.5, GPT-4, most modern LLMs |


## 4. Seeing it in the real world: `tiktoken`

`tiktoken` is OpenAI's production BPE tokenizer library — the same family of
tokenizers behind GPT-2 through GPT-4. It is **byte-level BPE**, exactly as
we just built by hand, just trained on a vastly larger corpus with a vocab
size in the tens of thousands instead of 8 toy merges.

Let's confirm this ourselves with real, adversarial inputs: emoji, Hindi
(Devanagari script), and math symbols — none of which were guaranteed to be
in any particular training set, yet none of which can ever cause a failure.


> **Note on running this section:** the two code cells below need internet access on first run, since `tiktoken` downloads its vocabulary files (e.g. `cl100k_base.tiktoken`) from OpenAI's servers the first time you use a given encoding, then caches it locally. If you're on a machine without outbound internet access, these two cells will raise a `403`/connection error — everything else in this notebook is fully self-contained and needs no network access at all. Run these two cells on your laptop or Colab to see the real output.

In [ ]:
import tiktoken

enc = tiktoken.get_encoding("cl100k_base")  # the GPT-3.5 / GPT-4 (base) tokenizer

test_strings = [
    "Hello, world!",
    "low, lower, lowest",
    "🙂🚀🔥",                      # emoji -- multi-byte UTF-8, never a single Unicode "character token"
    "नमस्ते दुनिया",                 # Hindi: "Hello world"
    "∑ x_i² ≠ ∫ f(x) dx",         # math symbols
]

for text in test_strings:
    token_ids = enc.encode(text)
    token_pieces = [enc.decode_single_token_bytes(t) for t in token_ids]
    print(f"Input text : {text!r}")
    print(f"Token IDs  : {token_ids}")
    print(f"Token bytes: {token_pieces}")
    print(f"Decodes back to original exactly? {enc.decode(token_ids) == text}")
    print("-" * 70)


Input text : 'Hello, world!'
Token IDs  : [9906, 11, 1917, 0]
Token bytes: [b'Hello', b',', b' world', b'!']
Decodes back to original exactly? True
----------------------------------------------------------------------
Input text : 'low, lower, lowest'
Token IDs  : [10516, 11, 4827, 11, 15821]
Token bytes: [b'low', b',', b' lower', b',', b' lowest']
Decodes back to original exactly? True
----------------------------------------------------------------------
Input text : '🙂🚀🔥'
Token IDs  : [9468, 19044, 9468, 248, 222, 9468, 242, 98]
Token bytes: [b'\xf0\x9f', b'\x99\x82', b'\xf0\x9f', b'\x9a', b'\x80', b'\xf0\x9f', b'\x94', b'\xa5']
Decodes back to original exactly? True
----------------------------------------------------------------------
Input text : 'नमस्ते दुनिया'
Token IDs  : [61196, 88344, 79468, 31584, 97, 35470, 15272, 99, 73753, 61196, 43411, 107, 24810]
Token bytes: [b'\xe0\xa4\xa8', b'\xe0\xa4\xae', b'\xe0\xa4\xb8', b'\xe0\xa5\x8d\xe0\xa4', b'\xa4', b'\xe0\xa5\x87', b' \xe0

Look closely at the `Token bytes` output for the emoji and Hindi lines.
Some tokens print as raw `b'\xe0\xa4...'` byte strings rather than clean
letters — that's the byte-level machinery peeking through. A single Hindi
character is often encoded as **multiple UTF-8 bytes**, and depending on
what merges the tokenizer learned, you may see those bytes grouped into one
token or split across a couple of tokens. Either way, **it never fails**,
and `decode(encode(text)) == text` holds for every single input, no matter
how exotic. That is the guarantee byte-level BPE buys you, and it is *why*
every modern LLM tokenizer uses it.


In [ ]:
print("Sanity check across MANY scripts and edge cases -- none of these were")
print("guaranteed to appear in training data, yet all of them succeed:\n")

edge_cases = [
    "",                       # empty string
    "😀" * 5,                 # repeated emoji
    "日本語のテキスト",           # Japanese
    "العربية",                # Arabic
    "🇮🇳",                     # flag emoji (a ZWJ / regional-indicator sequence -- tricky!)
    "\t\n  weird\u200bwhitespace",  # tabs, newlines, zero-width space
]

all_passed = True
for text in edge_cases:
    ids = enc.encode(text)
    ok = enc.decode(ids) == text
    all_passed &= ok
    print(f"{text!r:35} -> {len(ids):3} tokens -> round-trips correctly: {ok}")

print(f"\nAll edge cases round-tripped correctly: {all_passed}")
print("This is only possible because the base alphabet (256 bytes) is complete")
print("by construction -- it was never 'learned' from a training corpus at all.")


Sanity check across MANY scripts and edge cases -- none of these were
guaranteed to appear in training data, yet all of them succeed:

''                                  ->   0 tokens -> round-trips correctly: True
'😀😀😀😀😀'                             ->  10 tokens -> round-trips correctly: True
'日本語のテキスト'                          ->   8 tokens -> round-trips correctly: True
'العربية'                           ->   5 tokens -> round-trips correctly: True
'🇮🇳'                                ->   6 tokens -> round-trips correctly: True
'\t\n  weird\u200bwhitespace'       ->   6 tokens -> round-trips correctly: True

All edge cases round-tripped correctly: True
This is only possible because the base alphabet (256 bytes) is complete
by construction -- it was never 'learned' from a training corpus at all.


## 5. Why not just use a *bigger* character vocabulary?

A natural student question: *"Why not just pre-register every Unicode
character that exists, so character-level BPE also never hits `<UNK>`?"*

Two reasons this doesn't work in practice:


In [ ]:
num_unicode_code_points = 0x110000  # Unicode's total addressable space
num_byte_values = 256

print(f"Total possible Unicode code points : {num_unicode_code_points:,}")
print(f"Total possible byte values         : {num_byte_values}")
print(f"Ratio                              : {num_unicode_code_points / num_byte_values:,.0f}x larger")


Total possible Unicode code points : 1,114,112
Total possible byte values         : 256
Ratio                              : 4,352x larger


1. **Embedding table size.** Every token in the base vocabulary needs its
   own row in the model's embedding matrix. A base alphabet of ~1.1 million
   Unicode code points (most of which will basically never appear in your
   data) is enormously wasteful compared to a base alphabet of 256.
2. **New characters keep appearing.** Unicode itself grows every year (new
   emoji, new scripts). A model trained on today's Unicode standard could
   still be blindsided tomorrow. Bytes never grow — a byte is a byte, forever
   0–255.

Byte-level BPE gets a *complete, permanently-fixed, tiny* base alphabet, and
then lets the *learned merges* do all the work of building up meaningful
multi-byte chunks (which, for common languages, quickly converge to look
just like whole characters or common subwords anyway).


## 6. Summary — the one-sentence version

> **Byte-level BPE is the identical BPE merging algorithm as classic BPE —
> count adjacent pairs, merge the most frequent, repeat — run on top of raw
> UTF-8 bytes instead of Unicode characters, which guarantees a complete,
> fixed, 256-symbol base vocabulary and therefore eliminates the
> out-of-vocabulary / `<UNK>` problem entirely.**

Everything else you know about BPE (merge counting, greedy merge order,
vocabulary size as a hyperparameter, subword tokens) applies identically to
both versions.

### Exercises for students
1. In `train_bpe`, print the vocabulary size at every step and plot vocab
   size vs. merge step for both the character and byte experiments.
2. Extend `encode_char_bpe` so that instead of returning `None` on an
   unknown character, it inserts a literal `"<UNK>"` token and continues —
   then re-encode `"low🙂est"` and see how many characters got destroyed.
3. Try `tiktoken.get_encoding("o200k_base")` (the GPT-4o tokenizer) on the
   same `test_strings` above. Does it use *more* or *fewer* tokens for the
   Hindi and emoji examples than `cl100k_base`? What does that suggest about
   how its training data differed?
4. Modify `word_to_bytes_symbols` to print the raw byte values (as integers,
   not `bytes` objects) for the word `"café"` — notice `"é"` alone takes
   **2 bytes** in UTF-8. Why does that matter for how many BPE merges a
   single accented character might need?
